In [1]:
using LowLevelFEM

In [2]:
using LinearAlgebra
using SparseArrays
using IterativeSolvers

"""
    solve_linear_system(A, B; solver=:backslash, preconditioner=nothing,
                        reltol=sqrt(eps()), maxiter=size(A, 1))

Solve the linear system `A * X = B`.

`B` may contain one or multiple right-hand sides stored columnwise.
"""
function solve_linear_system(
    A,
    B;
    solver=:backslash,
    preconditioner=nothing,
    reltol=sqrt(eps()),
    maxiter=size(A, 1)
)
    if solver === :backslash
        return A \ B
    
    elseif solver === :auto_symmetric
        try
            F = cholesky(A)
            return F \ B
        catch err
            if err isa PosDefException
                @info "Cholesky factorization failed; falling back to LU."
                F = lu(sparse(A))
                return F \ B
            end
            rethrow()
        end

    elseif solver === :lu
        A0 =
            A isa Union{Symmetric,Hermitian} ?
            sparse(A) :
            A

        F = lu(A0)
        return F \ B

    elseif solver === :cholesky
        F = cholesky(A)
        return F \ B

    elseif solver === :qr
        F = qr(A)
        return F \ B

    elseif solver === :cg
        X = similar(B)

        for j in axes(B, 2)
            if preconditioner === nothing
                X[:, j] = cg(
                    A,
                    @view(B[:, j]);
                    reltol=reltol,
                    maxiter=maxiter
                )
            else
                X[:, j] = cg(
                    A,
                    @view(B[:, j]);
                    Pl=preconditioner,
                    reltol=reltol,
                    maxiter=maxiter
                )
            end
        end

        return X

    elseif solver === :gmres
        X = similar(B)

        for j in axes(B, 2)
            if preconditioner === nothing
                X[:, j] = gmres(
                    A,
                    @view(B[:, j]);
                    reltol=reltol,
                    maxiter=maxiter
                )
            else
                X[:, j] = gmres(
                    A,
                    @view(B[:, j]);
                    Pl=preconditioner,
                    reltol=reltol,
                    maxiter=maxiter
                )
            end
        end

        return X

    else
        error("Unknown linear solver: $solver.")
    end
end

solve_linear_system (generic function with 1 method)

In [3]:
"""
    default_solver(K)

Return the default linear solver associated with the matrix wrapper.
"""
default_solver(::SystemMatrix) = :backslash
default_solver(::LowLevelFEM.SymmetricSystemMatrix) = :auto_symmetric

default_solver (generic function with 2 methods)

In [4]:
"""
    prepare_singlefield_system(K, f, support)

Prepare a single-field finite element system for solution.

Returns the reduced free-DOF matrix, right-hand side matrix, prescribed
values, and DOF index sets.
"""
function prepare_singlefield_system(
    K::SystemMatrix,
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition}
)
    K.problems === nothing ||
        error("prepare_singlefield_system: K must be a single-field SystemMatrix.")

    P = K.model

    size(K.A, 1) == size(K.A, 2) ||
        error("prepare_singlefield_system: system matrix must be square.")

    size(f.a, 1) == size(K.A, 1) ||
        error("prepare_singlefield_system: incompatible matrix and vector sizes.")

    nsteps = size(f.a, 2)

    fixed = constrainedDoFs(P, support)
    free  = freeDoFs(P, support)

    xDfield = applyBoundaryConditions(
        P,
        support;
        steps=nsteps
    )

    xD = xDfield.a

    A_ff = K.A[free, free]

    B = copy(f.a[free, :])

    if !isempty(fixed)
        B .-= K.A[free, fixed] * xD[fixed, :]
    end

    return (
        A=A_ff,
        B=B,
        xD=xD,
        free=free,
        fixed=fixed
    )
end

prepare_singlefield_system (generic function with 1 method)

In [5]:
"""
    prepare_singlefield_system(Ks, f, support)

Prepare a symmetric single-field finite element system while preserving
the symmetric matrix representation.
"""
function prepare_singlefield_system(
    Ks::LowLevelFEM.SymmetricSystemMatrix,
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition}
)
    K = Ks.parent
    P = K.model

    K.problems === nothing ||
        error("prepare_singlefield_system: K must be a single-field SystemMatrix.")

    nsteps = size(f.a, 2)

    fixed = constrainedDoFs(P, support)
    free  = freeDoFs(P, support)

    xDfield = applyBoundaryConditions(
        P,
        support;
        steps=nsteps
    )

    xD = xDfield.a

    A = Symmetric(K.A, Ks.uplo)

    B = copy(f.a[free, :])

    if !isempty(fixed)
        B .-= A[free, fixed] * xD[fixed, :]
    end

    A_ff = Symmetric(
        K.A[free, free],
        Ks.uplo
    )

    return (
        A=A_ff,
        B=B,
        xD=xD,
        free=free,
        fixed=fixed
    )
end

prepare_singlefield_system (generic function with 2 methods)

In [6]:
function prepare_singlefield_system(
    K::SystemMatrix,
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition};
    mpc::Vector{MPC}=MPC[]
)
    return _prepare_singlefield_system(
        K,
        f,
        support;
        mpc=mpc
    )
end

prepare_singlefield_system (generic function with 2 methods)

In [7]:
function prepare_singlefield_system(
    K::LowLevelFEM.SymmetricSystemMatrix,
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition};
    mpc::Vector{MPC}=MPC[]
)
    return _prepare_singlefield_system(
        K,
        f,
        support;
        mpc=mpc
    )
end

prepare_singlefield_system (generic function with 2 methods)

In [8]:
"""
    reconstruct_singlefield_solution(f, Xfree, prepared)

Reconstruct the full finite element field from the free-DOF solution.
"""
function reconstruct_singlefield_solution(
    f::Union{ScalarField,VectorField,TensorField},
    Xfree,
    prepared
)
    u = copy(f)

    fill!(u.a, 0.0)

    u.a[prepared.free, :] .= Xfree

    if !isempty(prepared.fixed)
        u.a[prepared.fixed, :] .=
            prepared.xD[prepared.fixed, :]
    end

    return u
end

reconstruct_singlefield_solution (generic function with 1 method)

In [9]:
#=
"""
    solveField_new(K, f; support=[], solver=:auto, kwargs...)

Solve a single-field finite element system.
"""
function solveField_new(
    K::Union{SystemMatrix,LowLevelFEM.SymmetricSystemMatrix},
    f::Union{ScalarField,VectorField,TensorField};
    support::Vector{BoundaryCondition}=BoundaryCondition[],
    solver=:auto,
    preconditioner=nothing,
    reltol=sqrt(eps()),
    maxiter=size(K isa LowLevelFEM.SymmetricSystemMatrix ? K.parent.A : K.A, 1)
)
    prepared =
        prepare_singlefield_system(
            K,
            f,
            support
        )

    solver0 =
        solver === :auto ?
        default_solver(K) :
        solver

    Xfree =
        solve_linear_system(
            prepared.A,
            prepared.B;
            solver=solver0,
            preconditioner=preconditioner,
            reltol=reltol,
            maxiter=maxiter
        )

    return reconstruct_singlefield_solution(
        f,
        Xfree,
        prepared
    )
end
=#

In [10]:
function solveField_new(
    K::Union{SystemMatrix,LowLevelFEM.SymmetricSystemMatrix},
    f::Union{ScalarField,VectorField,TensorField};
    support::Vector{BoundaryCondition}=BoundaryCondition[],
    mpc::Vector{MPC}=MPC[],
    solver=:auto,
    preconditioner=nothing,
    reltol=sqrt(eps()),
    maxiter=size(K isa LowLevelFEM.SymmetricSystemMatrix ? K.parent.A : K.A, 1)
)
    prepared =
        prepare_singlefield_system(
            K,
            f,
            support;
            mpc=mpc
        )

    solver0 =
        solver === :auto ?
        default_solver(K) :
        solver

    Xfree =
        solve_linear_system(
            prepared.A,
            prepared.B;
            solver=solver0,
            preconditioner=preconditioner,
            reltol=reltol,
            maxiter=maxiter
        )

    return reconstruct_singlefield_solution(
        f,
        Xfree,
        prepared
    )
end

solveField_new (generic function with 1 method)

In [11]:
"""
    field_transformation(P)

Return the prolongation and restriction matrices associated with the field.

For a full-order field this is the active identity mapping.
For a reduced-order field the polynomial-order reduction is used.
"""
function field_transformation(P::Problem)
    if P.reducedOrder
        return reductionMatrices(P)
    else
        return LowLevelFEM.active_identity(P)
    end
end

field_transformation (generic function with 1 method)

In [12]:
"""
    field_transformation(P, mpc=MPC[])

Return the complete field transformation including polynomial-order
reduction and multi-point constraints.
"""
function field_transformation(
    P::Problem,
    mpc::Vector{MPC}=MPC[]
)
    return LowLevelFEM._singlefield_mpc_transformation(
        P,
        mpc
    )
end

field_transformation (generic function with 2 methods)

In [13]:
"""
    singlefield_mpc_bc_data_matrix(P, mpc, fixed, xD)

Transfer prescribed values from MPC slave DOFs to their final master DOFs
for one or multiple right-hand sides.
"""
function singlefield_mpc_bc_data_matrix(
    P::Problem,
    mpc::Vector{MPC},
    fixed::Vector{Int},
    xD::AbstractMatrix
)
    isempty(mpc) && return fixed, xD

    nsteps = size(xD, 2)

    xD_eff =
        zeros(
            eltype(xD),
            size(xD)
        )

    fixed_eff = Int[]

    for j in 1:nsteps

        fixed_j, xD_j =
            LowLevelFEM._singlefield_mpc_bc_data(
                P,
                mpc,
                fixed,
                @view xD[:, j]
            )

        if j == 1
            fixed_eff = fixed_j
        elseif fixed_j != fixed_eff
            error(
                "MPC: inconsistent constrained DOFs between right-hand sides."
            )
        end

        xD_eff[:, j] .= xD_j
    end

    return fixed_eff, xD_eff
end

singlefield_mpc_bc_data_matrix (generic function with 1 method)

In [14]:
"""
    reduced_bc_data_matrix(R, fixed, xD)

Map full-space Dirichlet data with one or multiple right-hand sides
to the reduced space.
"""
function reduced_bc_data_matrix(
    R::SparseMatrixCSC,
    fixed::AbstractVector{<:Integer},
    xD::AbstractMatrix
)
    nr = size(R, 1)

    if isempty(fixed)
        return (
            collect(1:nr),
            Int[],
            zeros(eltype(xD), nr, size(xD, 2))
        )
    end

    # The constrained reduced DOFs depend only on the set of
    # constrained full-space DOFs, not on their prescribed values.
    free_r, fixed_r, _ =
        reduced_bc_data(
            R,
            fixed,
            @view xD[:, 1]
        )

    xD_r = R * xD

    return free_r, fixed_r, xD_r
end

reduced_bc_data_matrix (generic function with 1 method)

In [15]:
parent_system(K::SystemMatrix) = K
parent_system(K::LowLevelFEM.SymmetricSystemMatrix) = K.parent

reduced_matrix_wrapper(::SystemMatrix, A) = A

reduced_matrix_wrapper(
    K::LowLevelFEM.SymmetricSystemMatrix,
    A
) = Symmetric(A, K.uplo)

reduced_matrix_wrapper (generic function with 2 methods)

In [16]:
"""
    prepare_singlefield_system(K, f, support)

Prepare a single-field finite element system for solution.

The same transformation pipeline is used for full-order and reduced-order
fields.
"""
function prepare_singlefield_system(
    K0::Union{SystemMatrix,LowLevelFEM.SymmetricSystemMatrix},
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition}
)
    K = parent_system(K0)
    P = K.model

    K.problems === nothing ||
        error(
            "prepare_singlefield_system: K must be a single-field SystemMatrix."
        )

    size(K.A, 1) == size(K.A, 2) ||
        error(
            "prepare_singlefield_system: system matrix must be square."
        )

    size(f.a, 1) == size(K.A, 1) ||
        error(
            "prepare_singlefield_system: incompatible matrix and vector sizes."
        )

    nsteps = size(f.a, 2)

    # ----------------------------------------------------------
    # 1) Full-space Dirichlet data
    # ----------------------------------------------------------

    fixed = constrainedDoFs(P, support)

    xDfield =
        applyBoundaryConditions(
            P,
            support;
            steps=nsteps
        )

    xD = xDfield.a

    # ----------------------------------------------------------
    # 2) Field transformation
    # ----------------------------------------------------------

    T, R = field_transformation(P)

    # ----------------------------------------------------------
    # 3) Dirichlet data in transformed space
    # ----------------------------------------------------------

    free_r, fixed_r, xD_r =
        reduced_bc_data_matrix(
            R,
            fixed,
            xD
        )

    # ----------------------------------------------------------
    # 4) Galerkin projection
    # ----------------------------------------------------------

    Kr = T' * K.A * T
    Br = T' * f.a

    # ----------------------------------------------------------
    # 5) Apply prescribed values
    # ----------------------------------------------------------

    B = copy(Br[free_r, :])

    if !isempty(fixed_r)
        B .-=
            Kr[free_r, fixed_r] *
            xD_r[fixed_r, :]
    end

    A =
        reduced_matrix_wrapper(
            K0,
            Kr[free_r, free_r]
        )

    return (
        A=A,
        B=B,
        T=T,
        xD_r=xD_r,
        free_r=free_r,
        fixed_r=fixed_r,
        nr=size(T, 2)
    )
end

prepare_singlefield_system (generic function with 3 methods)

In [17]:
"""
    reconstruct_singlefield_solution(f, Xfree, prepared)

Reconstruct the full finite element field from the transformed solution.
"""
function reconstruct_singlefield_solution(
    f::Union{ScalarField,VectorField,TensorField},
    Xfree,
    prepared
)
    nsteps = size(Xfree, 2)

    xr =
        zeros(
            eltype(Xfree),
            prepared.nr,
            nsteps
        )

    xr[prepared.free_r, :] .= Xfree

    if !isempty(prepared.fixed_r)
        xr[prepared.fixed_r, :] .=
            prepared.xD_r[prepared.fixed_r, :]
    end

    x = prepared.T * xr

    u = copy(f)
    u.a .= x

    return u
end

reconstruct_singlefield_solution (generic function with 1 method)

In [18]:
function _prepare_singlefield_system(
    K0,
    f,
    support;
    mpc::Vector{MPC}=MPC[]
)
    K = parent_system(K0)
    P = K.model

    K.problems === nothing ||
        error(
            "prepare_singlefield_system: K must be a single-field SystemMatrix."
        )

    size(K.A, 1) == size(K.A, 2) ||
        error(
            "prepare_singlefield_system: system matrix must be square."
        )

    size(f.a, 1) == size(K.A, 1) ||
        error(
            "prepare_singlefield_system: incompatible matrix and vector sizes."
        )

    nsteps = size(f.a, 2)

    # ----------------------------------------------------------
    # 1) Full-space Dirichlet data
    # ----------------------------------------------------------

    fixed = constrainedDoFs(P, support)

    xDfield =
        applyBoundaryConditions(
            P,
            support;
            steps=nsteps
        )

    xD = xDfield.a

    # ----------------------------------------------------------
    # 2) Complete field transformation
    #    full order / P1-P2 / MPC / combinations
    # ----------------------------------------------------------

    T, R =
        field_transformation(
            P,
            mpc
        )

    # ----------------------------------------------------------
    # 3) Transfer BC data through MPC relations
    # ----------------------------------------------------------

    fixed_eff, xD_eff =
        singlefield_mpc_bc_data_matrix(
            P,
            mpc,
            fixed,
            xD
        )

    # ----------------------------------------------------------
    # 4) Dirichlet data in transformed space
    # ----------------------------------------------------------

    free_r, fixed_r, xD_r =
        reduced_bc_data_matrix(
            R,
            fixed_eff,
            xD_eff
        )

    # ----------------------------------------------------------
    # 5) Galerkin projection
    # ----------------------------------------------------------

    Kr = T' * K.A * T
    Br = T' * f.a

    # ----------------------------------------------------------
    # 6) Apply prescribed values
    # ----------------------------------------------------------

    B = copy(Br[free_r, :])

    if !isempty(fixed_r)
        B .-=
            Kr[free_r, fixed_r] *
            xD_r[fixed_r, :]
    end

    A =
        reduced_matrix_wrapper(
            K0,
            Kr[free_r, free_r]
        )

    return (
        A=A,
        B=B,
        T=T,
        xD_r=xD_r,
        free_r=free_r,
        fixed_r=fixed_r,
        nr=size(T, 2)
    )
end

_prepare_singlefield_system (generic function with 1 method)

In [19]:
prepare_singlefield_system(
    K::SystemMatrix,
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition}
) =
    _prepare_singlefield_system(K, f, support)

prepare_singlefield_system (generic function with 3 methods)

In [20]:
prepare_singlefield_system(
    K::LowLevelFEM.SymmetricSystemMatrix,
    f::Union{ScalarField,VectorField,TensorField},
    support::Vector{BoundaryCondition}
) =
    _prepare_singlefield_system(K, f, support)

prepare_singlefield_system (generic function with 3 methods)

In [21]:
structured_box_mesh()

mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U))
f = ∫(U ⋅ [1,0,0], Γ="right")
support = [BoundaryCondition("left", ux=0.01, uy=0, uz=0)]

1-element Vector{BoundaryCondition}:
 BoundaryCondition("left", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :uz => 0, :ux => 0.01))

In [22]:
u_old = solveField(
    K,
    f;
    support=support
)

u_new = solveField_new(
    K,
    f;
    support=support
)

norm(u_old.a - u_new.a)

6.712186201863951e-16

In [23]:
us_old = solveField(
    Symmetric(K),
    f;
    support=support
)

us_new = solveField_new(
    Symmetric(K),
    f;
    support=support
)

norm(us_old.a - us_new.a)

2.8803466837580606e-16

In [24]:
u_lu = solveField_new(
    Symmetric(K),
    f;
    support=support,
    solver=:lu
)

u_chol = solveField_new(
    Symmetric(K),
    f;
    support=support,
    solver=:cholesky
)

(
    norm(u_new.a - u_lu.a),
    norm(u_new.a - u_chol.a)
)

(7.699625828096422e-16, 7.253952967820608e-16)

In [25]:
u_old = solveField(
    K,
    f;
    support=support
)

u_new = solveField_new(
    K,
    f;
    support=support
)

norm(u_old.a - u_new.a)

6.712186201863951e-16

In [26]:
us_old = solveField(
    Symmetric(K),
    f;
    support=support
)

us_new = solveField_new(
    Symmetric(K),
    f;
    support=support
)

norm(us_old.a - us_new.a)

2.8803466837580606e-16

In [27]:
structured_box_mesh(order=2)

mat = Material("body")
Ur = Field([mat], type=:VectorField, dim=3, fieldName=:u, reducedOrder=true)

Kr = ∫(ε(Ur) ⋅ D(:Solid, mat) ⋅ ε(Ur))
fr = ∫(Ur ⋅ [1,0,0], Γ="right")
support = [BoundaryCondition("left", ux=0.01, uy=0, uz=0)]

1-element Vector{BoundaryCondition}:
 BoundaryCondition("left", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :uz => 0, :ux => 0.01))

In [28]:
u_old = solveField(
    Kr,
    fr;
    support=support
)

u_new = solveField_new(
    Kr,
    fr;
    support=support
)

(
    size(u_old.a),
    size(u_new.a),
    norm(u_old.a - u_new.a)
)

((27783, 1), (27783, 1), 1.864866056738379e-15)

In [29]:
u_new_sym = solveField_new(
    Symmetric(Kr),
    fr;
    support=support
)

norm(u_old.a - u_new_sym.a)

2.5340501185153356e-15

In [30]:
T, R = reductionMatrices(Ur)

(
    size(Kr.A),
    size(T),
    size(R)
)

((27783, 27783), (27783, 3993), (3993, 27783))

In [31]:
prepared = prepare_singlefield_system(
    Kr,
    fr,
    support
)

(
    size(Kr.A),
    size(prepared.T),
    size(prepared.A)
)

((27783, 27783), (27783, 3993), (3630, 3630))

In [32]:
openGeometry("mpc-1.geo")

#openPreProcessor()
mat1 = Material("body")
mat2 = Material("remote")

U = Field([mat1, mat2], type=:VectorField, dim=2, fieldName=:u, rhsName=:f)
Ku = ∫(SymGrad(U) ⋅ [2 1 0; 1 2 0; 0 0 1] ⋅ SymGrad(U), Ω="body")
mpc = MPC(master="remote", slave="right", uy=false)
bc2 = BoundaryCondition("remote", uy=0)
bc = BoundaryCondition("left", ux=0, uy=0)

fu = ∫(U ⋅ [1, 0], Γ="remote")
u_old = solveField(
    Ku,
    fu;
    support=[bc, bc2],
    mpc=[mpc]
)

u_new = solveField_new(
    Ku,
    fu;
    support=[bc, bc2],
    mpc=[mpc]
)

(
    norm(u_old.a - u_new.a),
    norm(u_old.a - u_new.a) / norm(u_old.a)
)

(0.0, 0.0)

In [33]:
structured_box_mesh()

mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U))
f = ∫(U ⋅ [1,0,0], Γ="right")
support = [BoundaryCondition("left", ux=0.01, uy=0, uz=0)]
u_old = solveField(
    K,
    f;
    support=support
)

u_new = solveField_new(
    K,
    f;
    support=support
)

norm(u_old.a - u_new.a)
us_old = solveField(
    Symmetric(K),
    f;
    support=support
)

us_new = solveField_new(
    Symmetric(K),
    f;
    support=support
)

norm(us_old.a - us_new.a)
u_lu = solveField_new(
    Symmetric(K),
    f;
    support=support,
    solver=:lu
)

u_chol = solveField_new(
    Symmetric(K),
    f;
    support=support,
    solver=:cholesky
)

(
    norm(u_new.a - u_lu.a),
    norm(u_new.a - u_chol.a)
)

(7.699625828096422e-16, 7.253952967820608e-16)

In [34]:
structured_box_mesh()
Premote = gmsh.model.occ.addPoint(1, 0.5, 0.5)
gmsh.model.occ.synchronize()
gmsh.model.addPhysicalGroup(0, [Premote], -1, "remote")

gmsh.model.mesh.clear()
gmsh.model.mesh.generate(3)

In [35]:
mat = Material("body")
rem = Material("remote")
U = Field([mat,rem], type=:VectorField, dim=3, fieldName=:u)
Ku = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U), Ω="body")
fu = ∫(U ⋅ [1,0,0], Γ="right")

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [36]:
u_sym_old = solveField(
    Ku,
    fu;
    support=[bc, bc2],
    mpc=[mpc]
)

u_sym = solveField_new(
    Symmetric(Ku),
    fu;
    support=[bc, bc2],
    mpc=[mpc]
)

nonrem = constrainedDoFs(U, [BoundaryCondition("body", ux=0, uy=0, uz=0)])
norm(u_sym_old.a[nonrem] - u_sym.a[nonrem])

1.9127181651395164e-5

In [37]:
d = u_sym_old.a[nonrem] - u_sym.a[nonrem]

(
    abs_error = norm(d),
    rel_error = norm(d) / norm(u_sym_old.a[nonrem]),
    max_error = maximum(abs, d),
    old_norm = norm(u_sym_old.a[nonrem])
)

(abs_error = 1.9127181651395164e-5, rel_error = 0.17738402642962886, max_error = 5.242783870632143e-7, old_norm = 0.00010782922248629432)

In [38]:
P = Ku.model

fixed = constrainedDoFs(P, [bc, bc2])

uD = copy(fu)
fill!(uD.a, 0.0)
applyBoundaryConditions!(uD, [bc, bc2])

T_old, R_old =
    LowLevelFEM._singlefield_mpc_transformation(
        P,
        [mpc]
    )

fixed_old, uD_old =
    LowLevelFEM._singlefield_mpc_bc_data(
        P,
        [mpc],
        fixed,
        @view uD.a[:, 1]
    )

free_old, fixed_r_old, uD_r_old =
    LowLevelFEM.reduced_bc_data(
        R_old,
        fixed_old,
        uD_old
    )

Kr_old = T_old' * Ku.A * T_old
fr_old = T_old' * @view(fu.a[:, 1])

if !isempty(fixed_r_old)
    fr_old .-=
        Kr_old[:, fixed_r_old] *
        uD_r_old[fixed_r_old]
end

A_old = Kr_old[free_old, free_old]
b_old = fr_old[free_old];

In [39]:
prepared =
    prepare_singlefield_system(
        Ku,
        fu,
        [bc, bc2];
        mpc=[mpc]
    )

(A = sparse([1, 19, 20, 110, 111, 112, 263, 377, 378, 379  …  3481, 3482, 3483, 3484, 3506, 3507, 3508, 3509, 3510, 3511], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  3511, 3511, 3511, 3511, 3511, 3511, 3511, 3511, 3511, 3511], [4700.854700854701, -2136.752136752137, 1068.3760683760693, 320.51282051281964, -801.282051282052, 1068.3760683760697, -1709.4017094017106, 1602.5641025641035, -160.25641025640985, -1709.401709401711  …  -213.67521367521266, 1.8189894035458565e-12, 2.2737367544323206e-13, 4273.50427350427, -7.275957614183426e-12, 7.16227077646181e-12, 4273.504273504269, 1.0913936421275139e-11, 3.2514435588382185e-11, 37606.83760683755], 3511, 3511), B = [0.0; 0.0; … ; 0.0; 0.0;;], T = sparse([1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  3987, 3988, 3989, 3990, 3991, 3992, 3993, 3994, 3995, 3996], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  3745, 3746, 3747, 3748, 3749, 3750, 3751, 3752, 3753, 3754], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 3996,

In [40]:
(
    size_old = size(A_old),
    size_new = size(prepared.A),

    A_error = norm(A_old - prepared.A),
    b_error = norm(b_old - vec(prepared.B)),

    A_rel = norm(A_old - prepared.A) / norm(A_old),
    b_rel = norm(b_old - vec(prepared.B)) / norm(b_old)
)

(size_old = (3511, 3511), size_new = (3511, 3511), A_error = 0.0, b_error = 0.0, A_rel = 0.0, b_rel = 0.0)

In [41]:
u_old = solveField(
    Ku,
    fu;
    support=[bc, bc2],
    mpc=[mpc]
)

u_new = solveField_new(
    Ku,
    fu;
    support=[bc, bc2],
    mpc=[mpc]
)

u_chol = solveField_new(
    Symmetric(Ku),
    fu;
    support=[bc, bc2],
    mpc=[mpc]
)

nodal VectorField
[0.0; 0.0; … ; -7.039394537379703e-7; 4.748235526723528e-7;;]

In [42]:
nonrem = constrainedDoFs(
    U,
    [BoundaryCondition("body", ux=0, uy=0, uz=0)]
)

(
    old_new =
        norm(u_old.a[nonrem] - u_new.a[nonrem]) /
        norm(u_old.a[nonrem]),

    new_chol =
        norm(u_new.a[nonrem] - u_chol.a[nonrem]) /
        norm(u_new.a[nonrem])
)

(old_new = 0.06740728735301957, new_chol = 0.24517428412955225)

In [43]:
p_general = prepare_singlefield_system(
    Ku,
    fu,
    [bc, bc2];
    mpc=[mpc]
)

p_sym = prepare_singlefield_system(
    Symmetric(Ku),
    fu,
    [bc, bc2];
    mpc=[mpc]
)

(A = [4700.854700854701 0.0 … 0.0 0.0; 0.0 4700.854700854701 … 0.0 0.0; … ; 0.0 0.0 … 37606.837606837566 3.2514435588382185e-11; 0.0 0.0 … 3.2514435588382185e-11 37606.83760683755], B = [0.0; 0.0; … ; 0.0; 0.0;;], T = sparse([1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  3987, 3988, 3989, 3990, 3991, 3992, 3993, 3994, 3995, 3996], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  3745, 3746, 3747, 3748, 3749, 3750, 3751, 3752, 3753, 3754], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 3996, 3754), xD_r = [0.0; 0.0; … ; 0.0; 0.0;;], free_r = [3, 6, 9, 12, 13, 14, 15, 16, 17, 19  …  3745, 3746, 3747, 3748, 3749, 3750, 3751, 3752, 3753, 3754], fixed_r = [1, 2, 4, 5, 7, 8, 10, 11, 18, 20  …  500, 501, 503, 504, 506, 507, 509, 510, 512, 513], nr = 3754)

In [44]:
A_general = p_general.A
A_sym = sparse(p_sym.A)

(
    A_rel =
        norm(A_general - A_sym) /
        norm(A_general),

    B_rel =
        norm(p_general.B - p_sym.B) /
        norm(p_general.B)
)

(A_rel = 2.2659888256216365e-17, B_rel = 0.0)

In [45]:
x_general =
    solve_linear_system(
        p_general.A,
        p_general.B;
        solver=:backslash
    )

x_chol =
    solve_linear_system(
        p_sym.A,
        p_sym.B;
        solver=:cholesky
    )

(
    res_general =
        norm(p_general.A * x_general - p_general.B) /
        norm(p_general.B),

    res_chol =
        norm(p_sym.A * x_chol - p_sym.B) /
        norm(p_sym.B)
)

(res_general = 2.2049990004062148e-15, res_chol = 3.0580869330847397e-15)

## Coordinate-system layer — prototype

This section extends the current `solveField_new` pipeline with nodal coordinate-system transformations.

Semantics:

- ordinary load fields are interpreted in the local nodal coordinate systems defined by `coordSys`;
- `Global(f)` marks a load field whose components are given in the global Cartesian basis;
- essential boundary-condition components (`ux`, `uy`, `uz`) are interpreted in the local basis wherever a coordinate system is active;
- the returned solution field is always transformed back to the global Cartesian basis;
- coordinate-system matrices are generated internally; the user does not need to construct or manipulate `Q`.

For the first prototype, coordinate systems are supported only for full-order single-field vector problems. Combining `coordSys` with `reducedOrder=true` or MPCs is intentionally rejected until the transformation ordering is defined unambiguously.


In [46]:
const _SolveFieldType = Union{ScalarField,VectorField,TensorField}

"""
    Global(f)

Mark a load field as expressed in the global Cartesian coordinate system.

Unwrapped load fields are interpreted in the local nodal coordinate system
defined by `coordSys`.
"""
struct GlobalField{F}
    field::F
end

Global(f::_SolveFieldType) = GlobalField(f)


"""
    SolveRHS

Internal symbolic sum of load fields expressed in local and global bases.
"""
struct SolveRHS{F}
    local_part::Union{Nothing,F}
    global_part::Union{Nothing,F}
end


function _check_rhs_compatibility(a, b)
    typeof(a) === typeof(b) ||
        error("solveField: local and global load terms must have the same field type.")

    a.model === b.model ||
        error("solveField: local and global load terms must belong to the same Problem.")

    size(a.a) == size(b.a) ||
        error("solveField: local and global load terms have incompatible sizes.")

    return nothing
end


_merge_rhs_fields(::Nothing, b) = b
_merge_rhs_fields(a, ::Nothing) = a

function _merge_rhs_fields(a, b)
    _check_rhs_compatibility(a, b)
    return a + b
end


function Base.:+(a::_SolveFieldType, b::GlobalField)
    _check_rhs_compatibility(a, b.field)
    return SolveRHS(a, b.field)
end

Base.:+(a::GlobalField, b::_SolveFieldType) = b + a

function Base.:+(a::GlobalField, b::GlobalField)
    _check_rhs_compatibility(a.field, b.field)
    return SolveRHS(nothing, a.field + b.field)
end

function Base.:+(a::SolveRHS, b::_SolveFieldType)
    return SolveRHS(
        _merge_rhs_fields(a.local_part, b),
        a.global_part
    )
end

Base.:+(a::_SolveFieldType, b::SolveRHS) = b + a

function Base.:+(a::SolveRHS, b::GlobalField)
    return SolveRHS(
        a.local_part,
        _merge_rhs_fields(a.global_part, b.field)
    )
end

Base.:+(a::GlobalField, b::SolveRHS) = b + a

function Base.:+(a::SolveRHS, b::SolveRHS)
    return SolveRHS(
        _merge_rhs_fields(a.local_part, b.local_part),
        _merge_rhs_fields(a.global_part, b.global_part)
    )
end


"""
    CoordinateSystemDefinition(phName, vectors...; field=nothing)

Assign a nodal coordinate-system definition to a physical group.

This is a notebook-level wrapper around the current LowLevelFEM
`CoordinateSystem` type. In the final core API, `phName` and `field`
can be incorporated directly into the coordinate-system definition.
"""
struct CoordinateSystemDefinition
    phName::String
    system::LowLevelFEM.CoordinateSystem
    problem::Union{Problem,Nothing}
end

function CoordinateSystemDefinition(
    phName::String,
    vect1;
    field=nothing
)
    return CoordinateSystemDefinition(
        phName,
        LowLevelFEM.CoordinateSystem(vect1),
        field
    )
end

function CoordinateSystemDefinition(
    phName::String,
    vect1,
    vect2;
    field=nothing
)
    return CoordinateSystemDefinition(
        phName,
        LowLevelFEM.CoordinateSystem(vect1, vect2),
        field
    )
end


CoordinateSystemDefinition

In [47]:
_rhs_parts(f::_SolveFieldType) = (f, nothing)
_rhs_parts(f::GlobalField) = (nothing, f.field)
_rhs_parts(f::SolveRHS) = (f.local_part, f.global_part)

function _rhs_template(rhs)
    local_part, global_part = _rhs_parts(rhs)

    if local_part !== nothing
        return local_part
    elseif global_part !== nothing
        return global_part
    else
        error("solveField: empty right-hand side.")
    end
end


"""
    build_coordinate_transformation(P, coordSys)

Build the combined nodal basis transformation `Q`.

Each coordinate-system definition acts as identity outside its physical group.
Overlapping physical groups are rejected in this first implementation.
"""
function build_coordinate_transformation(
    P::Problem,
    coordSys
)
    isempty(coordSys) && return nothing

    P.pdim == P.dim ||
        error(
            "solveField: coordSys currently requires a vector field with pdim == dim."
        )

    P.dim in (2, 3) ||
        error(
            "solveField: coordSys currently supports only 2D and 3D vector fields."
        )

    gmsh.model.setCurrent(P.name)

    used_nodes = Set{Int}()

    Q = spdiagm(
        0 => ones(Float64, ndofs(P))
    )

    for cs in coordSys

        cs isa CoordinateSystemDefinition ||
            error(
                "solveField: every coordSys entry must be a CoordinateSystemDefinition."
            )

        if cs.problem !== nothing && cs.problem !== P
            error(
                "solveField: coordinate-system definition refers to a different Problem."
            )
        end

        tag =
            LowLevelFEM.getTagForPhysicalName(
                cs.phName
            )

        nodeTags, _ =
            gmsh.model.mesh.getNodesForPhysicalGroup(
                -1,
                tag
            )

        nodes = Set(Int.(nodeTags))
        overlap = intersect(used_nodes, nodes)

        isempty(overlap) ||
            error(
                "solveField: overlapping coordinate systems are not supported yet. " *
                "Physical group '$(cs.phName)' overlaps previously assigned nodes."
            )

        union!(used_nodes, nodes)

        Qi =
            LowLevelFEM.rotateNodes(
                P,
                cs.phName,
                cs.system
            ).T

        Q = Q * Qi
    end

    dropzeros!(Q)

    return Q
end


"""
    rhs_in_solver_coordinates(rhs, Q)

Return a template field and the RHS matrix expressed in solver coordinates.

Ordinary terms are local. Terms wrapped in `Global(...)` are transformed
from global Cartesian components by `Q'`.
"""
function rhs_in_solver_coordinates(
    rhs,
    Q
)
    local_part, global_part = _rhs_parts(rhs)
    template = _rhs_template(rhs)

    if local_part !== nothing && global_part !== nothing
        _check_rhs_compatibility(
            local_part,
            global_part
        )
    end

    B =
        zeros(
            eltype(template.a),
            size(template.a)
        )

    if local_part !== nothing
        B .+= local_part.a
    end

    if global_part !== nothing
        if Q === nothing
            B .+= global_part.a
        else
            B .+= Q' * global_part.a
        end
    end

    return template, B
end


rhs_in_solver_coordinates (generic function with 1 method)

In [48]:
"""
    _prepare_singlefield_system(K, rhs, support; mpc=[], coordSys=[])

Prepare a single-field system for the common solve pipeline.

Coordinate transformation is applied before the kinematic transformation:

    u_global = Q * u_local
    u_local  = T * u_reduced

so that

    K_solver = T' * Q' * K * Q * T.

In this first prototype, `coordSys` cannot yet be combined with MPC or
polynomial-order reduction.
"""
function _prepare_singlefield_system(
    K0,
    rhs,
    support;
    mpc::Vector{MPC}=MPC[],
    coordSys=CoordinateSystemDefinition[]
)
    K = parent_system(K0)
    P = K.model

    K.problems === nothing ||
        error(
            "prepare_singlefield_system: K must be a single-field SystemMatrix."
        )

    size(K.A, 1) == size(K.A, 2) ||
        error(
            "prepare_singlefield_system: system matrix must be square."
        )

    template = _rhs_template(rhs)

    template.model === P ||
        error(
            "prepare_singlefield_system: RHS and system matrix belong to different Problems."
        )

    size(template.a, 1) == size(K.A, 1) ||
        error(
            "prepare_singlefield_system: incompatible matrix and RHS sizes."
        )

    # ----------------------------------------------------------
    # 1) Coordinate-system transformation
    # ----------------------------------------------------------

    Q =
        build_coordinate_transformation(
            P,
            coordSys
        )

    if Q !== nothing
        P.reducedOrder &&
            error(
                "solveField: coordSys combined with reducedOrder=true is not supported yet."
            )

        !isempty(mpc) &&
            error(
                "solveField: coordSys combined with MPC is not supported yet."
            )

        template isa VectorField ||
            error(
                "solveField: coordSys currently supports VectorField problems only."
            )
    end

    rhs_template, F0 =
        rhs_in_solver_coordinates(
            rhs,
            Q
        )

    nsteps = size(F0, 2)

    A0 =
        Q === nothing ?
        K.A :
        Q' * K.A * Q

    # ----------------------------------------------------------
    # 2) Dirichlet data
    #
    # Components are interpreted directly in solver coordinates.
    # Thus ux/uy/uz are local on nodes affected by coordSys.
    # ----------------------------------------------------------

    fixed =
        constrainedDoFs(
            P,
            support
        )

    xDfield =
        applyBoundaryConditions(
            P,
            support;
            steps=nsteps
        )

    xD = xDfield.a

    # ----------------------------------------------------------
    # 3) Kinematic transformation
    #    full order / P1-P2 / MPC
    # ----------------------------------------------------------

    T, R =
        field_transformation(
            P,
            mpc
        )

    # ----------------------------------------------------------
    # 4) Transfer BC data through MPC relations
    # ----------------------------------------------------------

    fixed_eff, xD_eff =
        singlefield_mpc_bc_data_matrix(
            P,
            mpc,
            fixed,
            xD
        )

    # ----------------------------------------------------------
    # 5) Dirichlet data in transformed space
    # ----------------------------------------------------------

    free_r, fixed_r, xD_r =
        reduced_bc_data_matrix(
            R,
            fixed_eff,
            xD_eff
        )

    # ----------------------------------------------------------
    # 6) Galerkin projection
    # ----------------------------------------------------------

    Kr = T' * A0 * T
    Br = T' * F0

    # ----------------------------------------------------------
    # 7) Apply prescribed values
    # ----------------------------------------------------------

    B = copy(Br[free_r, :])

    if !isempty(fixed_r)
        B .-=
            Kr[free_r, fixed_r] *
            xD_r[fixed_r, :]
    end

    A =
        reduced_matrix_wrapper(
            K0,
            Kr[free_r, free_r]
        )

    return (
        A=A,
        B=B,
        T=T,
        Q=Q,
        template=rhs_template,
        xD_r=xD_r,
        free_r=free_r,
        fixed_r=fixed_r,
        nr=size(T, 2)
    )
end


function reconstruct_singlefield_solution(
    Xfree,
    prepared
)
    nsteps = size(Xfree, 2)

    xr =
        zeros(
            eltype(Xfree),
            prepared.nr,
            nsteps
        )

    xr[prepared.free_r, :] .= Xfree

    if !isempty(prepared.fixed_r)
        xr[prepared.fixed_r, :] .=
            prepared.xD_r[
                prepared.fixed_r,
                :
            ]
    end

    x_local =
        prepared.T * xr

    x_global =
        prepared.Q === nothing ?
        x_local :
        prepared.Q * x_local

    u = copy(prepared.template)
    u.a .= x_global

    return u
end


# Existing Field input: redefine the current notebook methods with coordSys support.
function prepare_singlefield_system(
    K::SystemMatrix,
    rhs::_SolveFieldType,
    support::Vector{BoundaryCondition};
    mpc::Vector{MPC}=MPC[],
    coordSys=CoordinateSystemDefinition[]
)
    return _prepare_singlefield_system(
        K,
        rhs,
        support;
        mpc=mpc,
        coordSys=coordSys
    )
end

function prepare_singlefield_system(
    K::LowLevelFEM.SymmetricSystemMatrix,
    rhs::_SolveFieldType,
    support::Vector{BoundaryCondition};
    mpc::Vector{MPC}=MPC[],
    coordSys=CoordinateSystemDefinition[]
)
    return _prepare_singlefield_system(
        K,
        rhs,
        support;
        mpc=mpc,
        coordSys=coordSys
    )
end


# Symbolic local/global RHS input.
function prepare_singlefield_system(
    K::SystemMatrix,
    rhs::Union{GlobalField,SolveRHS},
    support::Vector{BoundaryCondition};
    mpc::Vector{MPC}=MPC[],
    coordSys=CoordinateSystemDefinition[]
)
    return _prepare_singlefield_system(
        K,
        rhs,
        support;
        mpc=mpc,
        coordSys=coordSys
    )
end

function prepare_singlefield_system(
    K::LowLevelFEM.SymmetricSystemMatrix,
    rhs::Union{GlobalField,SolveRHS},
    support::Vector{BoundaryCondition};
    mpc::Vector{MPC}=MPC[],
    coordSys=CoordinateSystemDefinition[]
)
    return _prepare_singlefield_system(
        K,
        rhs,
        support;
        mpc=mpc,
        coordSys=coordSys
    )
end


prepare_singlefield_system (generic function with 5 methods)

In [49]:
function _solveField_new_impl(
    K,
    rhs;
    support::Vector{BoundaryCondition}=BoundaryCondition[],
    mpc::Vector{MPC}=MPC[],
    coordSys=CoordinateSystemDefinition[],
    solver=:auto,
    preconditioner=nothing,
    reltol=sqrt(eps()),
    maxiter=size(
        K isa LowLevelFEM.SymmetricSystemMatrix ?
        K.parent.A :
        K.A,
        1
    )
)
    prepared =
        prepare_singlefield_system(
            K,
            rhs,
            support;
            mpc=mpc,
            coordSys=coordSys
        )

    solver0 =
        solver === :auto ?
        default_solver(K) :
        solver

    Xfree =
        solve_linear_system(
            prepared.A,
            prepared.B;
            solver=solver0,
            preconditioner=preconditioner,
            reltol=reltol,
            maxiter=maxiter
        )

    return reconstruct_singlefield_solution(
        Xfree,
        prepared
    )
end


function solveField_new(
    K::Union{SystemMatrix,LowLevelFEM.SymmetricSystemMatrix},
    rhs::_SolveFieldType;
    support::Vector{BoundaryCondition}=BoundaryCondition[],
    mpc::Vector{MPC}=MPC[],
    coordSys=CoordinateSystemDefinition[],
    solver=:auto,
    preconditioner=nothing,
    reltol=sqrt(eps()),
    maxiter=size(
        K isa LowLevelFEM.SymmetricSystemMatrix ?
        K.parent.A :
        K.A,
        1
    )
)
    return _solveField_new_impl(
        K,
        rhs;
        support=support,
        mpc=mpc,
        coordSys=coordSys,
        solver=solver,
        preconditioner=preconditioner,
        reltol=reltol,
        maxiter=maxiter
    )
end


function solveField_new(
    K::Union{SystemMatrix,LowLevelFEM.SymmetricSystemMatrix},
    rhs::Union{GlobalField,SolveRHS};
    support::Vector{BoundaryCondition}=BoundaryCondition[],
    mpc::Vector{MPC}=MPC[],
    coordSys=CoordinateSystemDefinition[],
    solver=:auto,
    preconditioner=nothing,
    reltol=sqrt(eps()),
    maxiter=size(
        K isa LowLevelFEM.SymmetricSystemMatrix ?
        K.parent.A :
        K.A,
        1
    )
)
    return _solveField_new_impl(
        K,
        rhs;
        support=support,
        mpc=mpc,
        coordSys=coordSys,
        solver=solver,
        preconditioner=preconditioner,
        reltol=reltol,
        maxiter=maxiter
    )
end


solveField_new (generic function with 2 methods)

### Minimal coordinate-system test

The local first axis on `"right"` is chosen as global \(+y\). Therefore

```julia
f_local = ∫(U ⋅ [1, 0, 0], Γ="right")
```

interpreted through `coordSys=[cs]` should represent the same physical load as

```julia
Global(∫(U ⋅ [0, 1, 0], Γ="right"))
```

The returned displacement fields are both in global Cartesian components.


In [50]:
structured_box_mesh()

mat = Material("body")
Uc = Field(
    [mat],
    type=:VectorField,
    dim=3,
    fieldName=:u
)

Kc =
    ∫(
        ε(Uc) ⋅
        D(:Solid, mat) ⋅
        ε(Uc)
    )

support_c = [
    BoundaryCondition(
        "left",
        ux=0,
        uy=0,
        uz=0
    )
]

# Local e1 = global +y, local e2 = global -x, local e3 = global +z.
cs = CoordinateSystemDefinition(
    "right",
    [0.0, 1.0, 0.0],
    [-1.0, 0.0, 0.0];
    field=Uc
)

f_local =
    ∫(
        Uc ⋅ [1.0, 0.0, 0.0],
        Γ="right"
    )

f_global =
    ∫(
        Uc ⋅ [0.0, 1.0, 0.0],
        Γ="right"
    )

u_local =
    solveField_new(
        Kc,
        f_local;
        support=support_c,
        coordSys=[cs]
    )

u_global =
    solveField_new(
        Kc,
        Global(f_global);
        support=support_c,
        coordSys=[cs]
    )

(
    abs_error =
        norm(
            u_local.a -
            u_global.a
        ),

    rel_error =
        norm(
            u_local.a -
            u_global.a
        ) /
        norm(u_global.a)
)


(abs_error = 0.0, rel_error = 0.0)

### Mixed local and global loads

A typical use case is a surface load expressed in the local normal/tangential basis plus a body force expressed in global Cartesian coordinates:

```julia
f = f_surface + Global(f_gravity)

u = solveField_new(
    K,
    f;
    support=support,
    coordSys=[cs_surface]
)
```

Outside the physical groups covered by `coordSys`, the local basis is the global Cartesian basis because the combined transformation is identity there.


In [51]:
e1 = [1.0, 1.0, 0.0]
e2 = [-1.0, 1.0, 0.0]

cs45 = CoordinateSystemDefinition(
    "right",
    e1,
    e2;
    field=Uc
)

CoordinateSystemDefinition("right", CoordinateSystem([1.0, 1.0, 0.0], [-1.0, 1.0, 0.0], Function[LowLevelFEM.var"#f#f##1"(), LowLevelFEM.var"#f#f##1"(), LowLevelFEM.var"#f#f##1"()], Function[LowLevelFEM.var"#f#f##1"(), LowLevelFEM.var"#f#f##1"(), LowLevelFEM.var"#f#f##1"()], [0, 0, 0], [0, 0, 0]), Problem("structured_box", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1331, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false))

In [52]:
support45 = [
    BoundaryCondition(
        "left",
        ux=0,
        uy=0,
        uz=0
    ),
    BoundaryCondition(
        "right",
        ux=0
    )
]

2-element Vector{BoundaryCondition}:
 BoundaryCondition("left", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :uz => 0, :ux => 0))
 BoundaryCondition("right", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:ux => 0))

In [53]:
fg =
    ∫(
        Uc ⋅ [0.0, 1.0, 0.0],
        Γ="right"
    )

u45 =
    solveField_new(
        Kc,
        Global(fg);
        support=support45,
        coordSys=[cs45]
    )

nodal VectorField
[0.0; 0.0; … ; 4.504389977681652e-6; 4.1132177508609644e-7;;]

In [54]:
Q45 =
    build_coordinate_transformation(
        Uc,
        [cs45]
    )

u45_local = Q45' * u45.a

3993×1 Matrix{Float64}:
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
 -1.453250941496549e-22
  ⋮
 -3.4609436801196144e-6
  2.7341546188665705e-6
  5.917929655812398e-7
 -3.932628016956633e-6
  3.262139207122737e-6
  5.433478450453833e-7
 -4.369593317561656e-6
  3.838697334083013e-6
  4.818327813760797e-7
 -4.7646840254321065e-6
  4.504389977681652e-6
  4.1132177508609644e-7

In [55]:
tag =
    LowLevelFEM.getTagForPhysicalName(
        "right"
    )

nodes, _ =
    gmsh.model.mesh.getNodesForPhysicalGroup(
        -1,
        tag
    )

dofs_e1 =
    [
        3 * (Int(n) - 1) + 1
        for n in nodes
    ]

maximum(abs, u45_local[dofs_e1])

3.633708825098305e-22